In [1]:
import ase
from ase.calculators.orca import ORCA
from ase import Atoms
from ase.calculators.orca import OrcaProfile
import sys


# sys.path.append('/..' + os.getcwd())
from cc2cc.utils import gen_mole
name = "ICONF-SI5H12_4"

mol = gen_mole(
    name,
    "0",
    1,
    0,
    "cc-pVDZ",
    True,
    "gmtkn-cc-pVDZ",
)
# profile = OrcaProfile(command='${PATH}/orca/orca')

# h = Atoms(
#     "N2",
#     [(0, 0, 0), (0, 0, 1.6)],
#     calculator=ORCA(
#         profile=profile,
#         charge=0,
#         mult=1,
#         directory="water",
#         orcasimpleinput="CCSD cc-pVDZ",
#         orcablocks="%pal nprocs 1 end",
#     ),
# )

# au2ev = 27.21138602

# h.get_total_energy() / au2ev

Generate ICONF-SI5H12_4_0.0000
Extend 0 1 0.0000
original mol [['Si' 0.0 -2.06141387 -0.87292226]
 ['Si' 0.0 0.0 -2.02977282]
 ['Si' 0.0 2.06141387 -0.87292226]
 ['Si' 0.0 2.04066436 1.47296712]
 ['Si' 0.0 -2.04066436 1.47296712]
 ['H' 1.19799593 -2.82548452 -1.31822012]
 ['H' -1.19799593 -2.82548452 -1.31822012]
 ['H' -1.19676843 0.0 -2.91518357]
 ['H' 1.19676843 0.0 -2.91518357]
 ['H' 1.19799593 2.82548452 -1.31822012]
 ['H' -1.19799593 2.82548452 -1.31822012]
 ['H' 0.0 3.4404145 1.9722957]
 ['H' -1.20608553 1.35278084 1.99746897]
 ['H' 1.20608553 1.35278084 1.99746897]
 ['H' 0.0 -3.4404145 1.9722957]
 ['H' 1.20608553 -1.35278084 1.99746897]
 ['H' -1.20608553 -1.35278084 1.99746897]]
extend mol [['Si' 0.0 -2.06141387 -0.87292226]
 ['Si' 0.0 0.0 -2.02977282]
 ['Si' 0.0 2.06141387 -0.87292226]
 ['Si' 0.0 2.04066436 1.47296712]
 ['Si' 0.0 -2.04066436 1.47296712]
 ['H' 1.19799593 -2.82548452 -1.31822012]
 ['H' -1.19799593 -2.82548452 -1.31822012]
 ['H' -1.19676843 0.0 -2.91518357]
 ['H' 

In [ ]:
import json

molecular_xyz = ""
for i_atom, atom_info in enumerate(mol._atom):
    molecular_xyz += (
        f"{atom_info[0]:<6}\t{atom_info[1][0]:<16.10}\t{atom_info[1][1]:<16.10}\t{atom_info[1][2]:<16.10}"
        + "\n"
    )

with open(f"tmp_mol/{name}.inp", "w") as f:
    f.write(
        f"""! CCSD
%basis
  # read an externally specified orbital basis
  GTOName      = "cc-pvdz.1.orca"
end
%method
  FrozenCore FC_NONE      #No frozencore approximation
  WriteJSONPropertyfile True
end
%MDCI Density Unrelaxed
end
%pal nprocs {os.environ.get("OMP_NUM_THREADS")} end
%maxcore 120000
%coords
 CTyp   xyz     # the type of coordinates = xyz or internal
 Charge {mol.charge}       # the total charge of the molecule
 Mult   {mol.spin+1}        # the multiplicity = 2S+1
 Units  bohrs    # the unit of length = angs or bohrs

 # the subblock coords is for the actual coordinates
 # for CTyp=xyz
  coords
{molecular_xyz}end
end
"""
    )

os.system(f"~/orca/orca tmp_mol/{name}.inp > tmp_mol/{name}.out")

In [ ]:
with open(f"tmp_mol/{name}.property.json", "r") as f:
    data = json.load(f)["Geometry_1"]
    e_cc = data["MDCI_Energies"]["TOTALENERGY"]
    dipole_cc = None
    if isinstance(data["Dipole_Moment"], list):
        for json_i in data["Dipole_Moment"]:
            if json_i["PropertyName"] == "MDCI_Dipole_Moment":
                dipole_cc = np.array(json_i["DIPOLETOTAL"]).reshape(3)
    print(e_cc, dipole_cc)

-1452.285112750567 [-1.15568839e-12  4.98990016e-11  5.91556788e-02]


AttributeError: 'Mole' object has no attribute 'nproc'

In [4]:
import pyscf

mf = pyscf.scf.RHF(mol)
mf.max_cycle = 200
mf.diis_space = 12
mf.kernel()
mycc = pyscf.cc.CCSD(mf)
mycc.max_cycle = 200
_, t1, t2 = mycc.kernel()
dm1_cc = mycc.make_rdm1(ao_repr=True)
print(mycc.e_tot - e_cc)
print(
    pyscf.scf.hf.dip_moment(
        mol=mol,
        dm=dm1_cc,
        unit="A.U.",
    )
    - dipole_cc
)

# mdft = pyscf.scf.RKS(mol)
# mdft.xc = "b3lyp"
# mdft.max_cycle = 250
# mdft.kernel()



******** <class 'pyscf.scf.hf.RHF'> ********
method = RHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 12
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 200
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /tmp/tmpmd646k9z
max_memory 8000 MB (current use 782 MB)
Set gradient conv threshold to 3.16228e-05
Initial guess from minao.
init E= -1451.07808091879
  HOMO = -0.269126159616833  LUMO = 0.0562342157073508
cycle= 1 E= -1451.54981367224  delta_E= -0.472  |g|= 0.345  |ddm|= 5.91
  HOMO = -0.404341813569648  LUMO = 0.0706723013374714
cycle= 2 E= -1451.6161120328  delta_E= -0.0663  |g|= 0.101  |ddm|= 1.23
  HOMO = -0.37011644561744  LUMO = 0.106452123968255
cycle= 3 E= -1451.62275363891  delta_E= -0.00664  |g|= 0.0334  |ddm|= 0.337
  HOMO = -0.382868722294399  LUMO = 0.0983178788402769
cycle= 4 E= -1451.6234629252  delta_E= -0.000709  |g|= 0.005

<class 'pyscf.cc.ccsd.CCSD'> does not have attributes  converged


Init t2, MP2 energy = -1452.17563629117  E_corr(MP2) -0.552148667396369
Init E_corr(CCSD) = -0.552148667397011
cycle = 1  E_corr(CCSD) = -0.63164235951282  dE = -0.0794936921  norm(t1,t2) = 0.0810984
cycle = 2  E_corr(CCSD) = -0.652681393391489  dE = -0.0210390339  norm(t1,t2) = 0.026807
cycle = 3  E_corr(CCSD) = -0.66166558108355  dE = -0.00898418769  norm(t1,t2) = 0.0114969
cycle = 4  E_corr(CCSD) = -0.661889066092388  dE = -0.000223485009  norm(t1,t2) = 0.00235467
cycle = 5  E_corr(CCSD) = -0.661620188888602  dE = 0.000268877204  norm(t1,t2) = 0.000968957
cycle = 6  E_corr(CCSD) = -0.661620819522886  dE = -6.30634284e-07  norm(t1,t2) = 0.000184801
cycle = 7  E_corr(CCSD) = -0.66162541631024  dE = -4.59678735e-06  norm(t1,t2) = 5.9054e-05
cycle = 8  E_corr(CCSD) = -0.661625656134861  dE = -2.39824621e-07  norm(t1,t2) = 1.42673e-05
cycle = 9  E_corr(CCSD) = -0.661625068922103  dE = 5.87212758e-07  norm(t1,t2) = 4.90568e-06
cycle = 10  E_corr(CCSD) = -0.661625166207851  dE = -9.7285747